# 01 — Embeddings y búsqueda semántica aplicada a ATM

## Primer notebook del bloque LLM

En el bloque anterior entrenábamos nuestros propios modelos de Machine Learning.

A partir de ahora cambia el enfoque:

- utilizaremos **modelos preentrenados**;
- representaremos texto mediante **embeddings**;
- compararemos significados mediante distancia/similitud;
- construiremos una pequeña **búsqueda semántica ATM**;
- prepararemos el mecanismo de recuperación que más adelante utilizará un sistema **RAG**.

El objetivo de este notebook es entender físicamente el flujo:

```text
texto
  ↓
modelo de embeddings
  ↓
vector
  ↓
similitud
  ↓
documentos relevantes
```

Un embedding **no responde preguntas**. Solo representa el contenido en un espacio vectorial que permite comparar significado.

## 0. Dependencia

Desde la raíz del proyecto `ai-atm`, instala Sentence Transformers si todavía no lo tienes:

```bash
uv add sentence-transformers
```

El modelo utilizado en este notebook se descargará la primera vez que ejecutes la celda de carga y después quedará en la caché local.

Usaremos:

`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

Es un modelo multilingüe, por lo que podremos trabajar directamente en español y también experimentar con consultas en otros idiomas.

## 1. Importaciones

In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 120)

## 2. Nuestro pequeño corpus ATM

Imaginemos que estos textos proceden de incidencias, informes operacionales, documentación o mensajes internos.

Cada fila será un pequeño **documento recuperable**. En un RAG real podrían ser fragmentos de documentos mucho mayores.

In [ ]:
documents = pd.DataFrame([
    {"id": "ATM-001", "area": "capacity",
     "text": "La capacidad del sector se ha reducido por un incremento inesperado del tráfico aéreo."},
    {"id": "ATM-002", "area": "weather",
     "text": "Las tormentas en la zona oeste están obligando a desviar varias rutas y aumentando las demoras."},
    {"id": "ATM-003", "area": "surveillance",
     "text": "Se ha detectado una degradación temporal en la cobertura del radar secundario."},
    {"id": "ATM-004", "area": "communications",
     "text": "La red de comunicaciones tierra-tierra presenta pérdidas intermitentes de conectividad."},
    {"id": "ATM-005", "area": "staffing",
     "text": "La dotación disponible es inferior a la planificada y será necesario reducir la configuración del sector."},
    {"id": "ATM-006", "area": "flight_planning",
     "text": "El sistema de planificación de vuelos está procesando con retraso algunos planes de vuelo."},
    {"id": "ATM-007", "area": "capacity",
     "text": "La demanda prevista supera el número de aeronaves que el sector puede gestionar durante la próxima hora."},
    {"id": "ATM-008", "area": "weather",
     "text": "La presencia de cumulonimbos está generando desvíos y cambios frecuentes de nivel de vuelo."},
    {"id": "ATM-009", "area": "runway",
     "text": "Una de las pistas permanece temporalmente cerrada por inspección y la capacidad del aeropuerto se ha reducido."},
    {"id": "ATM-010", "area": "communications",
     "text": "Se observan dificultades en el intercambio de mensajes entre varios sistemas operacionales."},
    {"id": "ATM-011", "area": "flow_management",
     "text": "Se ha aplicado una regulación ATFM para limitar la entrada de tráfico en el sector congestionado."},
    {"id": "ATM-012", "area": "surveillance",
     "text": "La información de vigilancia presenta actualizaciones irregulares para algunas aeronaves."},
    {"id": "ATM-013", "area": "sector_configuration",
     "text": "Se ha abierto una configuración adicional de sectores para absorber el aumento de demanda."},
    {"id": "ATM-014", "area": "coordination",
     "text": "La carga de coordinación entre sectores adyacentes ha aumentado por múltiples reroutings."},
    {"id": "ATM-015", "area": "normal",
     "text": "Todos los sistemas operacionales funcionan con normalidad y el nivel de tráfico se mantiene estable."},
])

documents

## 3. Cargamos un modelo de embeddings

No vamos a entrenar este modelo. Ya ha sido entrenado para transformar textos en vectores de forma que textos con significado relacionado tiendan a quedar próximos en el espacio vectorial.

In [ ]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

model = SentenceTransformer(MODEL_NAME)

print(MODEL_NAME)

## 4. Convertimos los documentos en vectores

Para recuperación de información distinguimos:

- **documentos**: información que queremos poder recuperar;
- **queries**: preguntas o búsquedas del usuario.

Usaremos `encode_document()` para el corpus y `encode_query()` para las consultas.

In [ ]:
document_embeddings = model.encode_document(
    documents["text"].tolist(),
    normalize_embeddings=True,
)

print("Shape:", document_embeddings.shape)

El resultado tendrá una forma similar a `(15, 384)`: 15 documentos, cada uno representado por cientos de números.

In [ ]:
print("Documento:")
print(documents.loc[0, "text"])

print("\nPrimeras 12 dimensiones del embedding:")
print(document_embeddings[0][:12])

No debemos interpretar una dimensión individual como si tuviera un significado aislado. El significado está distribuido en el conjunto del vector.

## 5. Similitud coseno

In [ ]:
example_sentences = [
    "El sector tiene demasiado tráfico.",
    "Hay más aeronaves de las que podemos gestionar.",
    "El radar presenta una degradación de cobertura.",
]

example_embeddings = model.encode(
    example_sentences,
    normalize_embeddings=True,
)

similarity_matrix = cosine_similarity(
    example_embeddings,
    example_embeddings,
)

pd.DataFrame(
    similarity_matrix,
    index=example_sentences,
    columns=example_sentences,
).round(3)

Esperamos que las dos primeras frases tengan una similitud mayor entre sí que con la tercera, aunque no utilicen exactamente las mismas palabras.

# 6. Primera búsqueda semántica

In [ ]:
def semantic_search(query, top_k=5):
    query_embedding = model.encode_query(
        [query],
        normalize_embeddings=True,
    )

    similarities = cosine_similarity(
        query_embedding,
        document_embeddings,
    )[0]

    results = documents.copy()
    results["similarity"] = similarities

    return (
        results
        .sort_values("similarity", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

## 7. Consulta sin repetir las palabras del documento

Busquemos: **"Tenemos demasiados aviones para los recursos disponibles"**.

In [ ]:
query = "Tenemos demasiados aviones para los recursos disponibles"

semantic_search(query)

## 8. Más consultas ATM

In [ ]:
queries = [
    "Hay problemas para intercambiar información entre sistemas.",
    "El mal tiempo está obligando a cambiar las rutas.",
    "No tenemos suficiente personal para mantener la configuración prevista.",
    "La vigilancia de algunas aeronaves no se actualiza correctamente.",
]

for query in queries:
    print("=" * 100)
    print("QUERY:", query)
    display(
        semantic_search(query, top_k=3)[
            ["id", "area", "text", "similarity"]
        ].round({"similarity": 3})
    )

## 9. Keyword search frente a semantic search

Construiremos deliberadamente una búsqueda muy simple basada en coincidencia de palabras. No pretende ser un buscador léxico real; solo sirve para comparar conceptos.

In [ ]:
def tokenize(text):
    return set(
        re.findall(
            r"[a-záéíóúüñ]+",
            text.lower()
        )
    )

def keyword_search(query, top_k=5):
    query_tokens = tokenize(query)

    results = documents.copy()

    results["keyword_score"] = results["text"].apply(
        lambda text: len(
            query_tokens.intersection(
                tokenize(text)
            )
        )
    )

    return (
        results
        .sort_values("keyword_score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

In [ ]:
query = "Hay más aeronaves de las que podemos manejar"

print("KEYWORD SEARCH")
display(
    keyword_search(query, top_k=5)[
        ["id", "area", "text", "keyword_score"]
    ]
)

print("\nSEMANTIC SEARCH")
display(
    semantic_search(query, top_k=5)[
        ["id", "area", "text", "similarity"]
    ].round({"similarity": 3})
)

La conclusión no es que la búsqueda por palabras sea mala. En sistemas reales pueden combinarse búsqueda léxica, vectorial, híbrida, metadata y reranking. Aquí aislamos la idea de **búsqueda vectorial semántica**.

# 10. Experimento multilingüe

In [ ]:
query = "Thunderstorms are forcing aircraft to deviate from their planned routes"

semantic_search(query, top_k=5)[
    ["id", "area", "text", "similarity"]
].round({"similarity": 3})

La consulta está en inglés y los documentos en español. El objetivo es comprobar que un embedding multilingüe puede conservar relaciones semánticas entre idiomas.

# 11. Visualización 2D del espacio de embeddings

In [ ]:
pca = PCA(n_components=2)

embeddings_2d = pca.fit_transform(
    document_embeddings
)

plot_df = documents.copy()
plot_df["x"] = embeddings_2d[:, 0]
plot_df["y"] = embeddings_2d[:, 1]

plt.figure(figsize=(10, 7))
plt.scatter(plot_df["x"], plot_df["y"])

for _, row in plot_df.iterrows():
    plt.annotate(
        row["id"],
        (row["x"], row["y"]),
        xytext=(5, 5),
        textcoords="offset points",
    )

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Proyección 2D de embeddings ATM")
plt.show()

Esta proyección pierde información. La búsqueda real utiliza el embedding completo, no las dos dimensiones mostradas.

# 12. De búsqueda semántica a retrieval

In [ ]:
def retrieve(query, top_k=3):
    return semantic_search(query, top_k=top_k)[
        ["id", "area", "text", "similarity"]
    ]

query = "¿Qué problemas podrían explicar una reducción de capacidad?"

retrieved = retrieve(query, top_k=3)
retrieved.round({"similarity": 3})

## 13. Construimos el contexto

In [ ]:
def build_context(query, top_k=3):
    retrieved_docs = retrieve(query, top_k=top_k)

    context_parts = []

    for _, row in retrieved_docs.iterrows():
        context_parts.append(
            f"[{row['id']}] {row['text']}"
        )

    return "\n".join(context_parts)

query = "¿Por qué podría haberse reducido la capacidad?"

context = build_context(query)

print(context)

Todavía no tenemos RAG completo.

Hemos construido:

```text
pregunta
   ↓
embedding
   ↓
comparación con documentos
   ↓
TOP-K
   ↓
contexto
```

Para completar RAG faltará:

```text
pregunta + contexto
        ↓
       LLM
        ↓
respuesta
```

# 14. Una limitación importante

Que un documento tenga la similitud más alta **no significa que sea necesariamente correcto ni suficiente**.

Un sistema real debe afrontar, entre otros:

- documentos irrelevantes;
- información contradictoria;
- chunking;
- metadata;
- filtros;
- actualización de información;
- selección de `top_k`;
- calidad del modelo de embeddings;
- evaluación del retrieval.

# 15. Pequeño ejercicio

Añade al corpus al menos cinco documentos propios relacionados con ATM y crea tres consultas que:

1. expresen la misma idea usando palabras diferentes;
2. mezclen dos conceptos;
3. estén escritas en inglés.

Comprueba qué recupera el sistema.

# 16. Preguntas de comprobación

Al terminar deberías poder responder:

1. ¿Qué es un embedding?
2. ¿Por qué no interpretamos cada dimensión por separado?
3. ¿Qué significa que dos embeddings estén próximos?
4. ¿Qué mide cosine similarity?
5. ¿Por qué una búsqueda semántica puede encontrar un documento aunque la query no utilice las mismas palabras?
6. ¿Qué diferencia existe entre un embedding model y un LLM?
7. ¿Qué diferencia existe entre documentos y queries?
8. ¿Por qué calculamos los embeddings del corpus por adelantado?
9. ¿Qué significa `top_k`?
10. ¿Por qué el resultado con mayor similitud no garantiza una respuesta correcta?
11. ¿Qué parte de un futuro RAG hemos construido?
12. ¿Qué pieza falta para producir una respuesta en lenguaje natural?

# 17. Lo que hemos construido

```text
                 CORPUS ATM
                     │
              encode_document
                     │
                     ▼
              EMBEDDINGS
                     │
                     │
QUERY ──encode_query─┘
          │
          ▼
   similitud coseno
          │
          ▼
       TOP-K
          │
          ▼
  contexto relevante
```

## Siguiente paso

Ahora entraremos en:

```text
texto
  ↓
tokens
  ↓
Transformer
  ↓
LLM
  ↓
generación
```

Después podremos unir ambos mundos:

```text
retrieval + LLM = RAG
```